In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.models.transformer_baseline import (
    IndependentBandTransformerClassifier,
)

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Device: cuda
GPU: NVIDIA GeForce RTX 2050
CUDA version: 11.8


In [3]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [4]:
outputs_dir = PROJECT_ROOT / "outputs" / "salinas"

split_path = (
    outputs_dir / "salinas_spatial_split_seed42.npz"
)

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Test samples:", len(test_indices))

Train samples: 32337
Validation samples: 10952
Test samples: 10840


In [5]:
patch_size = 15

train_dataset = SalinasPatchDataset(
    scene,
    indices=train_indices,
    patch_size=patch_size,
)

val_dataset = SalinasPatchDataset(
    scene,
    indices=val_indices,
    patch_size=patch_size,
)

test_dataset = SalinasPatchDataset(
    scene,
    indices=test_indices,
    patch_size=patch_size,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 32337
Validation samples: 10952
Test samples: 10840


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

print("DataLoaders created.")

DataLoaders created.


In [7]:
sample_x, sample_y = train_dataset[0]

print("Sample patch:", sample_x.shape)
print("Sample label:", sample_y)

Sample patch: (204, 15, 15)
Sample label: 7


In [8]:
# Create the independent-band Transformer
#
# Salinas:
#   - 204 spectral bands
#   - 16 classes
#   - 15x15 input patches
#
# We use patch_size=15 to match the existing
# independent-band Transformer experiment.

model = IndependentBandTransformerClassifier(
    num_classes=len(SALINAS_CLASS_NAMES),
    patch_size=15,
    embed_dim=128,
    depth=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.1,
    max_bands=256,
    max_patches=4096,
).to(device)

print(model)

print(
    "Trainable parameters:",
    count_parameters(model)
)

IndependentBandTransformerClassifier(
  (patch_embed): IndependentBandPatchEmbed(
    (embed): Linear(in_features=225, out_features=128, bias=True)
    (band_embedding): Embedding(256, 128)
    (patch_embedding): Embedding(4096, 128)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine

In [9]:
# Check that the Transformer accepts our actual Salinas input

x, y = next(iter(train_loader))

print("Input:", x.shape)
print("Labels:", y.shape)

x = x.to(
    device=device,
    dtype=torch.float32,
)

with torch.no_grad():
    logits = model(x)

print("Output:", logits.shape)

Input: torch.Size([128, 204, 15, 15])
Labels: torch.Size([128])
Output: torch.Size([128, 16])


In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print("Loss:", criterion)
print("Optimizer:", optimizer)

Loss: CrossEntropyLoss()
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)


In [11]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for x, y in tqdm(
        loader,
        desc="Training",
        leave=False,
    ):
        x = x.to(
            device=device,
            dtype=torch.float32,
        )

        y = y.to(
            device=device,
            dtype=torch.long,
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() * x.shape[0]
        )

        total_samples += x.shape[0]

    return total_loss / total_samples

In [12]:
@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(
        loader,
        desc="Validation",
        leave=False,
    ):
        x = x.to(
            device=device,
            dtype=torch.float32,
        )

        logits = model(x)

        pred = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        all_pred.append(pred)
        all_true.append(y.numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_f1": macro_f1_score(
            y_true,
            y_pred,
            num_classes=len(SALINAS_CLASS_NAMES),
        ),
    }

In [13]:
epochs = 50
patience = 50

best_macro_f1 = -1.0
best_epoch = None
best_state = None

epochs_without_improvement = 0

history = []

transformer_checkpoint_path = (
    outputs_dir / "transformer_spatial_best.pt"
)

print(
    "Checkpoint:",
    transformer_checkpoint_path
)

Checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\transformer_spatial_best.pt


In [14]:
for epoch in range(1, epochs + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
    )

    val_metrics = evaluate(
        model,
        val_loader,
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1: {val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_macro_f1:

        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        torch.save(
            {
                "model_state_dict": best_state,
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_macro_f1,
                "class_names": SALINAS_CLASS_NAMES,
                "patch_size": patch_size,
                "bands": scene.bands,
                "seed": SEED,
            },
            transformer_checkpoint_path,
        )

        print(
            "✓ New best Transformer model — checkpoint saved"
        )

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:

        print(
            f"Early stopping at epoch {epoch}"
        )

        break

Epoch 01 | Loss: 0.3449 | Val Acc: 0.6768 | Val Macro-F1: 0.5728
✓ New best Transformer model — checkpoint saved


Epoch 02 | Loss: 0.0499 | Val Acc: 0.8661 | Val Macro-F1: 0.7952
✓ New best Transformer model — checkpoint saved


Epoch 03 | Loss: 0.0495 | Val Acc: 0.8364 | Val Macro-F1: 0.7846


Epoch 04 | Loss: 0.0209 | Val Acc: 0.8307 | Val Macro-F1: 0.8143
✓ New best Transformer model — checkpoint saved


Epoch 05 | Loss: 0.0383 | Val Acc: 0.8519 | Val Macro-F1: 0.7661


...

Epoch 46 | Loss: 0.0211 | Val Acc: 0.8031 | Val Macro-F1: 0.7979


Epoch 47 | Loss: 0.0337 | Val Acc: 0.8676 | Val Macro-F1: 0.8048


Epoch 48 | Loss: 0.0269 | Val Acc: 0.8535 | Val Macro-F1: 0.7571


Epoch 49 | Loss: 0.0173 | Val Acc: 0.8703 | Val Macro-F1: 0.8397
✓ New best Transformer model — checkpoint saved


Epoch 50 | Loss: 0.0088 | Val Acc: 0.8790 | Val Macro-F1: 0.7904


In [15]:
# Restore the best Transformer checkpoint found on validation Macro-F1

if best_state is None:
    raise RuntimeError("No best Transformer checkpoint was saved.")

model.load_state_dict(best_state)
model.to(device)
model.eval()

print("Restored best epoch:", best_epoch)
print("Best validation Macro-F1:", best_macro_f1)

Restored best epoch: 49
Best validation Macro-F1: 0.8397295740647048


In [16]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()

    all_true = []
    all_pred = []

    for x, y in tqdm(loader, desc="Collecting Transformer test predictions"):
        x = x.to(device=device, dtype=torch.float32)

        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()

        all_pred.append(pred)
        all_true.append(y.numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    return y_true, y_pred

In [17]:
y_test, y_pred = collect_predictions(
    model,
    test_loader
)

print("Number of test samples:", len(y_test))
print("Predictions:", len(y_pred))

Number of test samples: 10840
Predictions: 10840


In [18]:
test_accuracy = accuracy_score(y_test, y_pred)
test_macro_f1 = macro_f1_score(y_test, y_pred)

print("=" * 50)
print("FINAL TRANSFORMER SPATIAL TEST RESULTS")
print("=" * 50)
print(f"Accuracy : {test_accuracy:.4f}")
print(f"Macro-F1 : {test_macro_f1:.4f}")

FINAL TRANSFORMER SPATIAL TEST RESULTS
Accuracy : 0.9313
Macro-F1 : 0.8994


In [19]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    target_names=SALINAS_CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

transformer_report_df = pd.DataFrame(report).T

display(transformer_report_df)

,precision,recall,f1-score,support
Brocoli_green_weeds_1,0.990175,0.997800,0.993973,909.000000
Brocoli_green_weeds_2,1.000000,0.857430,0.923243,498.000000
Fallow,0.910769,1.000000,0.953301,296.000000
Fallow_rough_plow,0.896657,1.000000,0.945513,295.000000
Fallow_smooth,0.995365,0.995365,0.995365,863.000000
Stubble,0.966361,1.000000,0.982893,632.000000
Celery,0.681818,1.000000,0.810811,120.000000
Grapes_untrained,0.931311,0.921352,0.926305,2899.000000
Soil_vinyard_develop,0.986123,0.966015,0.975966,1177.000000
Corn_senesced_green_weeds,0.871728,0.612132,0.719222,544.000000


In [20]:
transformer_report_df.to_csv(
    outputs_dir / "transformer_spatial_per_class_metrics.csv"
)

print("Saved Transformer per-class metrics.")

Saved Transformer per-class metrics.


In [21]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(SALINAS_CLASS_NAMES)),
)

transformer_cm_df = pd.DataFrame(
    cm,
    index=SALINAS_CLASS_NAMES,
    columns=SALINAS_CLASS_NAMES,
)

display(transformer_cm_df)

,Brocoli_green_weeds_1,Brocoli_green_weeds_2,Fallow,Fallow_rough_plow,Fallow_smooth,Stubble,Celery,Grapes_untrained,Soil_vinyard_develop,Corn_senesced_green_weeds,Lettuce_romaine_4wk,Lettuce_romaine_5wk,Lettuce_romaine_6wk,Lettuce_romaine_7wk,Vinyard_untrained,Vinyard_vertical_trellis
Brocoli_green_weeds_1,907,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2
Brocoli_green_weeds_2,9,427,0,0,0,9,53,0,0,0,0,0,0,0,0,0
Fallow,0,0,296,0,0,0,0,0,0,0,0,0,0,0,0,0
Fallow_rough_plow,0,0,0,295,0,0,0,0,0,0,0,0,0,0,0,0
Fallow_smooth,0,0,0,4,859,0,0,0,0,0,0,0,0,0,0,0
Stubble,0,0,0,0,0,632,0,0,0,0,0,0,0,0,0,0
Celery,0,0,0,0,0,0,120,0,0,0,0,0,0,0,0,0
Grapes_untrained,0,0,0,0,4,1,0,2671,0,23,9,0,0,1,190,0
Soil_vinyard_develop,0,0,1,0,0,12,0,6,1137,8,0,10,3,0,0,0
Corn_senesced_green_weeds,0,0,28,0,0,0,3,54,16,333,84,23,0,3,0,0


In [22]:
transformer_cm_df.to_csv(
    outputs_dir / "transformer_spatial_confusion_matrix.csv"
)

print("Saved Transformer confusion matrix.")

Saved Transformer confusion matrix.


In [23]:
np.save(
    outputs_dir / "transformer_spatial_y_test.npy",
    y_test,
)

np.save(
    outputs_dir / "transformer_spatial_y_pred.npy",
    y_pred,
)

print("Saved Transformer predictions.")

Saved Transformer predictions.


In [24]:
transformer_results = pd.DataFrame([
    {
        "model": "Transformer",
        "test_accuracy": test_accuracy,
        "test_macro_f1": test_macro_f1,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_macro_f1,
    }
])

display(transformer_results)

transformer_results.to_csv(
    outputs_dir / "transformer_spatial_results.csv",
    index=False,
)

,model,test_accuracy,test_macro_f1,best_epoch,best_val_macro_f1
0,Transformer,0.931273,0.899447,49,0.83973


In [25]:
comparison = pd.DataFrame([
    {
        "model": "CNN",
        "test_accuracy": 0.960148,
        "test_macro_f1": 0.941450,
    },
    {
        "model": "Hybrid Spatial-Spectral",
        "test_accuracy": 0.980996,
        "test_macro_f1": 0.963905,
    },
    {
        "model": "Transformer",
        "test_accuracy": test_accuracy,
        "test_macro_f1": test_macro_f1,
    },
])

display(comparison)

,model,test_accuracy,test_macro_f1
0,CNN,0.960148,0.941450
1,Hybrid Spatial-Spectral,0.980996,0.963905
2,Transformer,0.931273,0.899447


In [26]:
comparison.to_csv(
    outputs_dir / "cnn_hybrid_transformer_spatial_comparison.csv",
    index=False,
)